# AI-Assisted Real Estate Underwriting Tool

**Andrew Brown** | Deal Routes

---

### How to use this

1. **Run the setup cell** (Section 1) once.
2. **Edit the deal box** in Section 2 — that is the only place you type anything.
3. **Run All.** Everything below generates automatically.

You get: a full cash flow model, IRR / equity multiple / DSCR, a sensitivity grid, an
investment memo, and an Excel + Word export package.

To underwrite **many deals at once**, skip to Section 7.

*This notebook is self-contained — no other files required.*


## 0. Install Check

Run this first. It installs anything missing and tells you if you're good to go.


In [ ]:
import importlib, subprocess, sys

needed = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
    "docx": "python-docx",     # optional, for the Word memo
}

missing = []
for module, pipname in needed.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(pipname)

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done. Now go to Kernel > Restart Kernel, then Run All.")
else:
    print("All packages present. Continue to Section 1.")


## 1. Setup & Model Engine

Run once. **Nothing to edit here** — this is the underwriting math.
Collapse this cell if it's in your way (click the blue bar to its left).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# =====================================================================
# UNDERWRITING ENGINE - no need to read or edit this
# =====================================================================

import numpy as np
import pandas as pd
from scipy.optimize import brentq


DEFAULTS = {
    "rent_growth": 0.03,
    "hold_period_years": 5,
    "selling_costs_pct": 0.02,
    "ltv": 0.60,
    "interest_rate": 0.055,
    "amort_years": 30,
}

REQUIRED = ["deal_name", "purchase_price", "going_in_noi", "exit_cap_rate"]


def _clean(deal: dict) -> dict:
    d = {**DEFAULTS, **deal}
    missing = [k for k in REQUIRED if k not in d or d[k] in (None, "")]
    if missing:
        raise ValueError(f"Missing required input(s): {', '.join(missing)}")
    return d


def _irr(cfs):
    def npv(r):
        return sum(cf / (1 + r) ** t for t, cf in enumerate(cfs))
    try:
        return brentq(npv, -0.99, 5.0)
    except ValueError:
        return float("nan")


def underwrite(deal: dict) -> dict:
    """Run one deal. Returns a dict of results plus a tidy cash flow table."""
    d = _clean(deal)
    n = int(d["hold_period_years"])
    yrs = np.arange(1, n + 1)

    noi = pd.Series(d["going_in_noi"] * (1 + d["rent_growth"]) ** (yrs - 1), index=yrs)
    exit_noi = noi.iloc[-1] * (1 + d["rent_growth"])
    terminal = exit_noi / d["exit_cap_rate"] * (1 - d["selling_costs_pct"])

    # Unlevered
    unlev = noi.copy()
    unlev.iloc[-1] += terminal
    unlev_cfs = [-d["purchase_price"]] + unlev.tolist()

    out = {
        "inputs": d,
        "noi": noi,
        "terminal_value": terminal,
        "going_in_cap": d["going_in_noi"] / d["purchase_price"],
        "unlevered_irr": _irr(unlev_cfs),
        "unlevered_multiple": sum(unlev_cfs[1:]) / d["purchase_price"],
        "unlevered_cfs": unlev_cfs,
    }

    # Levered
    if d["ltv"] > 0:
        loan = d["purchase_price"] * d["ltv"]
        mr = d["interest_rate"] / 12
        npmt = int(d["amort_years"] * 12)
        pmt = loan * (mr * (1 + mr) ** npmt) / ((1 + mr) ** npmt - 1)
        ads = pmt * 12

        bal = loan
        principal_by_yr = []
        for _ in range(n):
            yr_prin = 0.0
            for _m in range(12):
                interest = bal * mr
                prin = pmt - interest
                bal -= prin
                yr_prin += prin
            principal_by_yr.append(yr_prin)
        remaining = loan - np.cumsum(principal_by_yr)

        lev = noi - ads
        lev.iloc[-1] += terminal - remaining[-1]
        equity = d["purchase_price"] - loan
        lev_cfs = [-equity] + lev.tolist()

        out.update({
            "loan_amount": loan,
            "equity_invested": equity,
            "annual_debt_service": ads,
            "dscr_year1": noi.iloc[0] / ads,
            "levered_irr": _irr(lev_cfs),
            "levered_multiple": sum(lev_cfs[1:]) / equity,
            "levered_cfs": lev_cfs,
        })

    # Tidy cash flow table for display/export
    tbl = pd.DataFrame({"Year": [f"Year {i}" for i in yrs], "NOI": noi.values})
    tbl["Terminal Value"] = [0.0] * (n - 1) + [terminal]
    tbl["Unlevered CF"] = unlev.values
    if d["ltv"] > 0:
        tbl["Debt Service"] = [-out["annual_debt_service"]] * n
        tbl["Levered CF"] = lev.values
    out["cash_flow_table"] = tbl
    return out


def sensitivity(deal: dict, price_mults=None, cap_deltas=None, metric="levered_irr"):
    """Grid of IRR outcomes across purchase price and exit cap scenarios."""
    d = _clean(deal)
    price_mults = price_mults or [0.95, 0.975, 1.0, 1.025, 1.05]
    cap_deltas = cap_deltas or [-0.0075, -0.00375, 0.0, 0.00375, 0.0075]
    if d["ltv"] == 0:
        metric = "unlevered_irr"

    rows = []
    for pm in price_mults:
        row = []
        for cd in cap_deltas:
            scen = {**d,
                    "purchase_price": d["purchase_price"] * pm,
                    "exit_cap_rate": d["exit_cap_rate"] + cd}
            row.append(underwrite(scen)[metric])
        rows.append(row)

    return pd.DataFrame(
        rows,
        index=[f"{pm:.1%} of ask" for pm in price_mults],
        columns=[f"{d['exit_cap_rate'] + cd:.2%}" for cd in cap_deltas],
    )


def portfolio(deals: list) -> pd.DataFrame:
    """Run many deals at once (e.g. loaded from CSV) and rank them."""
    recs = []
    for deal in deals:
        r = underwrite(deal)
        recs.append({
            "Deal": r["inputs"]["deal_name"],
            "Purchase Price": r["inputs"]["purchase_price"],
            "Going-In Cap": r["going_in_cap"],
            "Exit Cap": r["inputs"]["exit_cap_rate"],
            "Unlevered IRR": r["unlevered_irr"],
            "Levered IRR": r.get("levered_irr", float("nan")),
            "Equity Multiple": r.get("levered_multiple", r["unlevered_multiple"]),
            "Yr1 DSCR": r.get("dscr_year1", float("nan")),
        })
    df = pd.DataFrame(recs)
    sort_col = "Levered IRR" if df["Levered IRR"].notna().any() else "Unlevered IRR"
    return df.sort_values(sort_col, ascending=False).reset_index(drop=True)


def memo_facts(results: dict) -> str:
    """Compact factual block passed to the LLM. Numbers only - no prose invented here."""
    d = results["inputs"]
    lines = [
        f"Deal: {d['deal_name']}",
        f"Purchase price: ${d['purchase_price']:,.0f}",
        f"Year 1 NOI: ${d['going_in_noi']:,.0f}",
        f"Going-in cap rate: {results['going_in_cap']:.2%}",
        f"NOI growth assumption: {d['rent_growth']:.2%} per year",
        f"Hold period: {d['hold_period_years']} years",
        f"Exit cap rate: {d['exit_cap_rate']:.2%}",
        f"Terminal value (net of {d['selling_costs_pct']:.1%} selling costs): ${results['terminal_value']:,.0f}",
        f"Unlevered IRR: {results['unlevered_irr']:.2%}",
        f"Unlevered equity multiple: {results['unlevered_multiple']:.2f}x",
    ]
    if d["ltv"] > 0:
        lines += [
            f"Leverage: {d['ltv']:.0%} LTV at {d['interest_rate']:.2%}, {d['amort_years']}-yr amortization",
            f"Loan amount: ${results['loan_amount']:,.0f}",
            f"Equity invested: ${results['equity_invested']:,.0f}",
            f"Annual debt service: ${results['annual_debt_service']:,.0f}",
            f"Year 1 DSCR: {results['dscr_year1']:.2f}x",
            f"Levered IRR: {results['levered_irr']:.2%}",
            f"Levered equity multiple: {results['levered_multiple']:.2f}x",
        ]
    return "\n".join(lines)


def memo_template(results: dict) -> str:
    """Offline memo - works with no API key."""
    d = results["inputs"]
    lev = d["ltv"] > 0
    irr_v = results["levered_irr"] if lev else results["unlevered_irr"]
    mult = results["levered_multiple"] if lev else results["unlevered_multiple"]

    memo = f"""INVESTMENT MEMORANDUM
{d['deal_name']}

RECOMMENDATION SUMMARY
Proposed acquisition of {d['deal_name']} at ${d['purchase_price']:,.0f}, a {results['going_in_cap']:.2%} \
going-in capitalization rate on Year 1 net operating income of ${d['going_in_noi']:,.0f}. \
Base-case underwriting projects a {irr_v:.2%} {'levered' if lev else 'unlevered'} internal rate of \
return and a {mult:.2f}x equity multiple over a {d['hold_period_years']}-year hold.

UNDERWRITING ASSUMPTIONS
Net operating income is grown at {d['rent_growth']:.2%} annually. Disposition is underwritten at a \
{d['exit_cap_rate']:.2%} exit capitalization rate applied to forward NOI, net of {d['selling_costs_pct']:.1%} \
selling costs, producing a terminal value of ${results['terminal_value']:,.0f}."""

    if lev:
        memo += f""" The capital structure assumes {d['ltv']:.0%} loan-to-value at {d['interest_rate']:.2%} \
interest on a {d['amort_years']}-year amortization schedule, requiring ${results['equity_invested']:,.0f} \
of equity against a ${results['loan_amount']:,.0f} loan. Year 1 debt service coverage is \
{results['dscr_year1']:.2f}x."""

    memo += """

KEY RISKS
Returns are most sensitive to exit capitalization rate and achieved disposition pricing; refer to the \
attached sensitivity analysis for IRR outcomes across the pricing and exit-cap range. Principal risks \
include underperformance against the NOI growth assumption, cap rate expansion at exit, and \
interest rate exposure at refinancing.

NEXT STEPS
Quantitative base case is complete. Qualitative diligence — market comparables, tenant credit and \
rollover schedule, physical condition assessment, and submarket supply pipeline — remains outstanding \
prior to a final recommendation."""
    return memo

print("Engine loaded. Continue to Section 2.")


## 2. Deal Inputs

**This is the only cell you edit.**

Required: `deal_name`, `purchase_price`, `going_in_noi`, `exit_cap_rate`.
Everything else falls back to a sensible default if you delete the line.
Set `ltv` to `0` for an all-cash / unlevered deal.


In [ ]:
DEAL = {
    # --- required ---
    "deal_name":          "Example Office Building",
    "purchase_price":     20_000_000,
    "going_in_noi":        1_200_000,     # Year 1 NOI
    "exit_cap_rate":       0.0625,        # 6.25%

    # --- optional (defaults shown) ---
    "rent_growth":         0.03,          # 3% annual NOI growth
    "hold_period_years":   5,
    "selling_costs_pct":   0.02,          # 2% of sale price
    "ltv":                 0.60,          # 60% leverage; use 0 for all-cash
    "interest_rate":       0.055,
    "amort_years":         30,
}

results = underwrite(DEAL)
print("Model run complete.")


## 3. Headline Metrics

In [ ]:
d = results["inputs"]
levered = d["ltv"] > 0

summary = {
    "Purchase Price":      f"${d['purchase_price']:,.0f}",
    "Year 1 NOI":          f"${d['going_in_noi']:,.0f}",
    "Going-In Cap Rate":   f"{results['going_in_cap']:.2%}",
    "Exit Cap Rate":       f"{d['exit_cap_rate']:.2%}",
    "Terminal Value":      f"${results['terminal_value']:,.0f}",
    "Unlevered IRR":       f"{results['unlevered_irr']:.2%}",
    "Unlevered Multiple":  f"{results['unlevered_multiple']:.2f}x",
}
if levered:
    summary.update({
        "Loan Amount":       f"${results['loan_amount']:,.0f}",
        "Equity Invested":   f"${results['equity_invested']:,.0f}",
        "Annual Debt Svc":   f"${results['annual_debt_service']:,.0f}",
        "Year 1 DSCR":       f"{results['dscr_year1']:.2f}x",
        "Levered IRR":       f"{results['levered_irr']:.2%}",
        "Levered Multiple":  f"{results['levered_multiple']:.2f}x",
    })

print(f"{d['deal_name'].upper()}")
print("=" * 46)
for k, v in summary.items():
    print(f"{k:<22}{v:>22}")


## 4. Cash Flow Projection

In [ ]:
cf = results["cash_flow_table"]
display_cf = cf.copy()
for col in display_cf.columns[1:]:
    display_cf[col] = display_cf[col].map(lambda x: f"${x:,.0f}")
display_cf


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(cf))
ax.bar([i - 0.2 for i in x], cf["NOI"], width=0.4, label="NOI", color="#1F4E78")
if levered:
    ax.bar([i + 0.2 for i in x], cf["Levered CF"], width=0.4, label="Levered CF", color="#7FB3D5")
ax.set_xticks(list(x))
ax.set_xticklabels(cf["Year"])
ax.set_ylabel("$")
ax.set_title(f"{d['deal_name']} — Annual Cash Flows")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Sensitivity Analysis

Rebuilds the entire model across 25 price / exit-cap combinations. Rows are what you pay
relative to the ask; columns are the exit cap rate.


In [ ]:
grid = sensitivity(DEAL)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(grid.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(grid.columns)))
ax.set_xticklabels(grid.columns)
ax.set_yticks(range(len(grid.index)))
ax.set_yticklabels(grid.index)
ax.set_xlabel("Exit Cap Rate")
ax.set_ylabel("Purchase Price")

for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        ax.text(j, i, f"{grid.values[i, j]:.1%}", ha="center", va="center", fontsize=9)

metric = "Levered" if levered else "Unlevered"
ax.set_title(f"{d['deal_name']} — {metric} IRR Sensitivity")
fig.colorbar(im, ax=ax, label="IRR")
plt.tight_layout()
plt.show()


## 6. Investment Memo

Two paths:

- **`memo_template()`** — works offline, no API key. Deterministic, always available.
- **`memo_ai()`** — sends only the computed figures to Claude and asks for IC-ready prose.
  Requires an `ANTHROPIC_API_KEY`. The model is explicitly instructed not to invent numbers,
  and the fact block it receives is printed below so you can audit exactly what was sent.


In [ ]:
memo = memo_template(results)
print(memo)


In [ ]:
# What gets sent to the model - audit this before trusting any AI output
print(memo_facts(results))


In [ ]:
import os

def memo_ai(results, model="claude-sonnet-4-6"):
    """Draft an IC memo from the computed figures. Returns None if no API key."""
    try:
        import anthropic
    except ImportError:
        print("Optional. To enable: pip install anthropic")
        return None

    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        print("No ANTHROPIC_API_KEY set - using the offline template above instead.")
        return None

    facts = memo_facts(results)
    prompt = f"""You are a real estate investment analyst drafting an internal
investment committee memo.

Use ONLY the figures below. Do not invent any number, comparable, tenant, or market fact
that is not listed. Where qualitative diligence would be required, say so explicitly rather
than inventing a conclusion.

Structure: Recommendation Summary / Underwriting Assumptions / Returns / Key Risks / Next Steps.
Professional, concise, under 400 words.

FIGURES:
{facts}"""

    client = anthropic.Anthropic(api_key=key)
    msg = client.messages.create(
        model=model,
        max_tokens=900,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text


ai_memo = memo_ai(results)
if ai_memo:
    print(ai_memo)


## 7. Batch Mode — Screen Many Deals

Underwrites a whole list of deals at once and ranks them. Edit the list below, or load
from a CSV / database query with the same field names.


In [ ]:
# Option A: edit this list directly
DEAL_LIST = [
    {"deal_name": "Example Office Building", "purchase_price": 20_000_000,
     "going_in_noi": 1_200_000, "exit_cap_rate": 0.0625, "ltv": 0.60},
    {"deal_name": "Desert Commons", "purchase_price": 14_500_000,
     "going_in_noi": 942_500, "exit_cap_rate": 0.065, "rent_growth": 0.028, "ltv": 0.65},
    {"deal_name": "3990 Ruffin Place", "purchase_price": 31_000_000,
     "going_in_noi": 1_953_000, "exit_cap_rate": 0.06, "rent_growth": 0.032,
     "hold_period_years": 7, "ltv": 0.55},
]

# Option B: load from a CSV instead (uncomment)
# DEAL_LIST = pd.read_csv("deals_template.csv").to_dict("records")

ranked = portfolio(DEAL_LIST)

styled = ranked.copy()
styled["Purchase Price"] = styled["Purchase Price"].map(lambda x: f"${x:,.0f}")
for c in ["Going-In Cap", "Exit Cap", "Unlevered IRR", "Levered IRR"]:
    styled[c] = styled[c].map(lambda x: f"{x:.2%}")
styled["Equity Multiple"] = styled["Equity Multiple"].map(lambda x: f"{x:.2f}x")
styled["Yr1 DSCR"] = styled["Yr1 DSCR"].map(lambda x: f"{x:.2f}x")
styled


## 8. Export Deliverable Package

Writes a formatted Excel workbook and a Word memo into an `output/` folder.

The workbook has six tabs — Executive Summary, Assumptions, Cash Flow Model, Sensitivity,
Deal Screen, and Investment Memo — and is a **live model**: the yellow cells on the
Assumptions tab are inputs, and everything downstream is a real Excel formula that
recalculates when you change them.


In [ ]:
from pathlib import Path

# =====================================================================
# EXCEL EXPORT - formatted, formula-driven workbook
# The output is a LIVE model: change an assumption cell in Excel and
# every downstream figure recalculates.
# =====================================================================

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

FONT = "Arial"

NAVY = "1F3864"
LIGHT = "D9E2F3"
GREY = "F2F2F2"

H1 = Font(name=FONT, size=16, bold=True, color="FFFFFF")
H2 = Font(name=FONT, size=11, bold=True, color="FFFFFF")
LBL = Font(name=FONT, size=10)
LBL_B = Font(name=FONT, size=10, bold=True)
INPUT = Font(name=FONT, size=10, color="0000FF")      # blue = hardcoded input
CALC = Font(name=FONT, size=10, color="000000")       # black = formula
NOTE = Font(name=FONT, size=9, italic=True, color="7F7F7F")
BIG = Font(name=FONT, size=14, bold=True, color=NAVY)

FILL_NAVY = PatternFill("solid", fgColor=NAVY)
FILL_LIGHT = PatternFill("solid", fgColor=LIGHT)
FILL_GREY = PatternFill("solid", fgColor=GREY)
FILL_INPUT = PatternFill("solid", fgColor="FFF2CC")

_t = Side(style="thin", color="BFBFBF")
_m = Side(style="medium", color=NAVY)
BOX = Border(left=_t, right=_t, top=_t, bottom=_t)
TOPLINE = Border(top=_m)

CUR0 = '$#,##0;($#,##0);"-"'
CUR2 = '$#,##0.00;($#,##0.00);"-"'
PCT1 = '0.0%'
PCT2 = '0.00%'
MULT = '0.00"x"'


def _title_bar(ws, text, subtitle, width=8):
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=width)
    c = ws.cell(row=1, column=1, value=text)
    c.font = H1
    c.fill = FILL_NAVY
    c.alignment = Alignment(vertical="center", horizontal="left", indent=1)
    ws.row_dimensions[1].height = 28

    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=width)
    s = ws.cell(row=2, column=2 - 1, value=subtitle)
    s.font = NOTE
    s.alignment = Alignment(indent=1)


def _section(ws, row, text, width=8):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=width)
    c = ws.cell(row=row, column=1, value=text)
    c.font = H2
    c.fill = FILL_NAVY
    c.alignment = Alignment(vertical="center", indent=1)
    ws.row_dimensions[row].height = 18


def build_workbook(results, grid, ranked, memo_text, path):
    d = results["inputs"]
    lev = d["ltv"] > 0
    n = int(d["hold_period_years"])

    wb = openpyxl.Workbook()

    # =================================================================
    # ASSUMPTIONS  (built first - everything else references it)
    # =================================================================
    a = wb.active
    a.title = "Assumptions"
    a.sheet_view.showGridLines = False
    a.column_dimensions["A"].width = 3
    a.column_dimensions["B"].width = 34
    a.column_dimensions["C"].width = 18
    a.column_dimensions["D"].width = 46

    _title_bar(a, d["deal_name"], "Underwriting assumptions  |  blue cells are inputs - edit these", width=5)

    _section(a, 4, "PROPERTY & PRICING", width=5)
    rows_prop = [
        ("Purchase Price", d["purchase_price"], CUR0, "Contract / ask price"),
        ("Year 1 Net Operating Income", d["going_in_noi"], CUR0, "In-place NOI from rent roll"),
    ]
    _section_rows(a, 5, rows_prop)

    a["B7"] = "Going-In Cap Rate"
    a["B7"].font = LBL_B
    a["C7"] = "=C6/C5"
    a["C7"].font = CALC
    a["C7"].number_format = PCT2
    a["C7"].border = BOX
    a["D7"] = "Calculated: Year 1 NOI / Purchase Price"
    a["D7"].font = NOTE

    _section(a, 9, "GROWTH & EXIT", width=5)
    rows_growth = [
        ("Annual NOI Growth", d["rent_growth"], PCT1, "Applied to NOI each year"),
        ("Hold Period (Years)", n, '0', "Years to disposition"),
        ("Exit Cap Rate", d["exit_cap_rate"], PCT2, "Applied to forward (Year N+1) NOI"),
        ("Selling Costs", d["selling_costs_pct"], PCT1, "Broker fees + closing costs at sale"),
    ]
    _section_rows(a, 10, rows_growth)

    _section(a, 15, "CAPITAL STRUCTURE", width=5)
    rows_debt = [
        ("Loan to Value (LTV)", d["ltv"], PCT1, "Set to 0% for an all-cash deal"),
        ("Interest Rate", d["interest_rate"], PCT2, "Fixed rate"),
        ("Amortization (Years)", d["amort_years"], '0', "Fully amortizing"),
    ]
    _section_rows(a, 16, rows_debt)

    a["B20"] = "Loan Amount"
    a["B20"].font = LBL_B
    a["C20"] = "=C5*C16"
    a["C20"].font = CALC
    a["C20"].number_format = CUR0
    a["C20"].border = BOX

    a["B21"] = "Equity Required"
    a["B21"].font = LBL_B
    a["C21"] = "=C5-C20"
    a["C21"].font = CALC
    a["C21"].number_format = CUR0
    a["C21"].border = BOX

    a["B22"] = "Annual Debt Service"
    a["B22"].font = LBL_B
    a["C22"] = "=IF(C20=0,0,C20*((C17/12)*(1+C17/12)^(C18*12))/((1+C17/12)^(C18*12)-1)*12)"
    a["C22"].font = CALC
    a["C22"].number_format = CUR0
    a["C22"].border = BOX
    a["D22"] = "Standard amortizing payment, annualized"
    a["D22"].font = NOTE

    # cell refs used by other sheets
    R = {
        "price": "Assumptions!$C$5",
        "noi1": "Assumptions!$C$6",
        "growth": "Assumptions!$C$10",
        "hold": "Assumptions!$C$11",
        "exitcap": "Assumptions!$C$12",
        "sell": "Assumptions!$C$13",
        "ltv": "Assumptions!$C$16",
        "loan": "Assumptions!$C$20",
        "equity": "Assumptions!$C$21",
        "ads": "Assumptions!$C$22",
        "rate": "Assumptions!$C$17",
        "amort": "Assumptions!$C$18",
    }

    # =================================================================
    # CASH FLOW MODEL  (years across columns, banker convention)
    # =================================================================
    cfw = wb.create_sheet("Cash Flow Model")
    cfw.sheet_view.showGridLines = False
    cfw.column_dimensions["A"].width = 3
    cfw.column_dimensions["B"].width = 32
    for i in range(n + 1):
        cfw.column_dimensions[get_column_letter(3 + i)].width = 15

    _title_bar(cfw, f"{d['deal_name']} - Cash Flow Model",
               "All figures calculated live from the Assumptions tab", width=3 + n)

    hdr = 4
    cfw.cell(row=hdr, column=2, value="").fill = FILL_NAVY
    cfw.cell(row=hdr, column=3, value="At Close").font = H2
    cfw.cell(row=hdr, column=3).fill = FILL_NAVY
    cfw.cell(row=hdr, column=3).alignment = Alignment(horizontal="center")
    for i in range(1, n + 1):
        c = cfw.cell(row=hdr, column=3 + i, value=f"Year {i}")
        c.font = H2
        c.fill = FILL_NAVY
        c.alignment = Alignment(horizontal="center")

    def put(row, label, bold=False, top=False):
        c = cfw.cell(row=row, column=2, value=label)
        c.font = LBL_B if bold else LBL
        if top:
            c.border = TOPLINE
        return c

    # NOI
    put(6, "Net Operating Income")
    for i in range(1, n + 1):
        col = get_column_letter(3 + i)
        prev = get_column_letter(2 + i)
        f = f"={R['noi1']}" if i == 1 else f"={prev}6*(1+{R['growth']})"
        c = cfw[f"{col}6"]
        c.value = f
        c.font = CALC
        c.number_format = CUR0

    # Terminal value
    put(7, "Net Sale Proceeds")
    endc = get_column_letter(3 + n)
    cfw[f"{endc}7"] = f"=({endc}6*(1+{R['growth']}))/{R['exitcap']}*(1-{R['sell']})"
    cfw[f"{endc}7"].font = CALC
    cfw[f"{endc}7"].number_format = CUR0

    # Unlevered
    put(9, "UNLEVERED CASH FLOW", bold=True, top=True)
    cfw["C9"] = f"=-{R['price']}"
    cfw["C9"].font = CALC
    cfw["C9"].number_format = CUR0
    cfw["C9"].border = TOPLINE
    for i in range(1, n + 1):
        col = get_column_letter(3 + i)
        c = cfw[f"{col}9"]
        c.value = f"={col}6+{col}7" if i == n else f"={col}6"
        c.font = CALC
        c.number_format = CUR0
        c.border = TOPLINE

    if lev:
        put(11, "Debt Service")
        for i in range(1, n + 1):
            col = get_column_letter(3 + i)
            c = cfw[f"{col}11"]
            c.value = f"=-{R['ads']}"
            c.font = CALC
            c.number_format = CUR0

        put(12, "Loan Balance Repaid at Exit")
        # remaining balance after n years of amortization
        cfw[f"{endc}12"] = (
            f"=-({R['loan']}*(1+{R['rate']}/12)^({R['hold']}*12)"
            f"-{R['ads']}/12/({R['rate']}/12)*((1+{R['rate']}/12)^({R['hold']}*12)-1))"
        )
        cfw[f"{endc}12"].font = CALC
        cfw[f"{endc}12"].number_format = CUR0

        put(14, "LEVERED CASH FLOW", bold=True, top=True)
        cfw["C14"] = f"=-{R['equity']}"
        cfw["C14"].font = CALC
        cfw["C14"].number_format = CUR0
        cfw["C14"].border = TOPLINE
        for i in range(1, n + 1):
            col = get_column_letter(3 + i)
            c = cfw[f"{col}14"]
            c.value = f"={col}9+{col}11+{col}12" if i == n else f"={col}9+{col}11"
            c.font = CALC
            c.number_format = CUR0
            c.border = TOPLINE

        put(16, "Debt Service Coverage Ratio")
        for i in range(1, n + 1):
            col = get_column_letter(3 + i)
            c = cfw[f"{col}16"]
            c.value = f"={col}6/{R['ads']}"
            c.font = CALC
            c.number_format = MULT

    # returns block
    rr = 19 if lev else 12
    _section(cfw, rr, "RETURNS", width=3 + n)
    end_col = get_column_letter(3 + n)

    cfw.cell(row=rr + 1, column=2, value="Unlevered IRR").font = LBL_B
    cfw.cell(row=rr + 1, column=3, value=f"=IRR(C9:{end_col}9)").number_format = PCT2
    cfw.cell(row=rr + 1, column=3).font = CALC

    cfw.cell(row=rr + 2, column=2, value="Unlevered Equity Multiple").font = LBL_B
    cfw.cell(row=rr + 2, column=3, value=f"=SUM(D9:{end_col}9)/-C9").number_format = MULT
    cfw.cell(row=rr + 2, column=3).font = CALC

    if lev:
        cfw.cell(row=rr + 3, column=2, value="Levered IRR").font = LBL_B
        cfw.cell(row=rr + 3, column=3, value=f"=IRR(C14:{end_col}14)").number_format = PCT2
        cfw.cell(row=rr + 3, column=3).font = CALC

        cfw.cell(row=rr + 4, column=2, value="Levered Equity Multiple").font = LBL_B
        cfw.cell(row=rr + 4, column=3, value=f"=SUM(D14:{end_col}14)/-C14").number_format = MULT
        cfw.cell(row=rr + 4, column=3).font = CALC

    # =================================================================
    # EXECUTIVE SUMMARY  (references the model - built after)
    # =================================================================
    s = wb.create_sheet("Executive Summary", 0)
    s.sheet_view.showGridLines = False
    s.column_dimensions["A"].width = 3
    s.column_dimensions["B"].width = 30
    s.column_dimensions["C"].width = 20
    s.column_dimensions["D"].width = 6
    s.column_dimensions["E"].width = 26

    _title_bar(s, d["deal_name"], "Investment Summary  |  prepared with the automated underwriting tool", width=5)

    _section(s, 4, "THE DEAL", width=5)
    pairs = [
        ("Purchase Price", f"={R['price']}", CUR0),
        ("Year 1 NOI", f"={R['noi1']}", CUR0),
        ("Going-In Cap Rate", "=Assumptions!$C$7", PCT2),
        ("Hold Period", f"={R['hold']}", '0" years"'),
        ("Exit Cap Rate", f"={R['exitcap']}", PCT2),
    ]
    for i, (lbl, f, fmt) in enumerate(pairs):
        r = 5 + i
        s.cell(row=r, column=2, value=lbl).font = LBL
        c = s.cell(row=r, column=3, value=f)
        c.font = CALC
        c.number_format = fmt
        c.border = BOX

    _section(s, 11, "RETURNS", width=5)
    ret_row = rr
    rets = [
        ("Unlevered IRR", f"='Cash Flow Model'!C{ret_row+1}", PCT2),
        ("Unlevered Equity Multiple", f"='Cash Flow Model'!C{ret_row+2}", MULT),
    ]
    if lev:
        rets += [
            ("Levered IRR", f"='Cash Flow Model'!C{ret_row+3}", PCT2),
            ("Levered Equity Multiple", f"='Cash Flow Model'!C{ret_row+4}", MULT),
            ("Year 1 DSCR", "='Cash Flow Model'!D16", MULT),
        ]
    for i, (lbl, f, fmt) in enumerate(rets):
        r = 12 + i
        s.cell(row=r, column=2, value=lbl).font = LBL_B
        c = s.cell(row=r, column=3, value=f)
        c.font = CALC
        c.number_format = fmt
        c.fill = FILL_LIGHT
        c.border = BOX

    # headline callout
    head_r = 12
    s.cell(row=head_r, column=5, value="HEADLINE RETURN").font = LBL_B
    hc = s.cell(row=head_r + 1, column=5,
                value=f"='Cash Flow Model'!C{ret_row + (3 if lev else 1)}")
    hc.font = BIG
    hc.number_format = PCT2
    hc.alignment = Alignment(horizontal="left")
    s.cell(row=head_r + 2, column=5,
           value=("Levered IRR" if lev else "Unlevered IRR") + f", {n}-year hold").font = NOTE

    if lev:
        _section(s, 18, "CAPITAL STRUCTURE", width=5)
        cap = [
            ("Loan Amount", f"={R['loan']}", CUR0),
            ("Equity Required", f"={R['equity']}", CUR0),
            ("Annual Debt Service", f"={R['ads']}", CUR0),
        ]
        for i, (lbl, f, fmt) in enumerate(cap):
            r = 19 + i
            s.cell(row=r, column=2, value=lbl).font = LBL
            c = s.cell(row=r, column=3, value=f)
            c.font = CALC
            c.number_format = fmt
            c.border = BOX

    note_r = 23 if lev else 18
    _section(s, note_r, "BASIS & LIMITATIONS", width=5)
    limits = [
        "NOI grows at a single constant rate; no explicit lease-up, rollover, or capex schedule.",
        "Terminal value is a forward-NOI capitalization, net of selling costs.",
        "Debt is fixed-rate and fully amortizing; no interest-only period or refinancing.",
        "No tax treatment, partnership waterfall, or promote structure.",
        "Screening-level analysis. A deal clearing this screen still requires a full property-level model.",
    ]
    for i, t in enumerate(limits):
        s.merge_cells(start_row=note_r + 1 + i, start_column=2, end_row=note_r + 1 + i, end_column=5)
        c = s.cell(row=note_r + 1 + i, column=2, value="- " + t)
        c.font = NOTE

    # =================================================================
    # SENSITIVITY
    # =================================================================
    sv = wb.create_sheet("Sensitivity")
    sv.sheet_view.showGridLines = False
    sv.column_dimensions["A"].width = 3
    sv.column_dimensions["B"].width = 20
    for i in range(len(grid.columns)):
        sv.column_dimensions[get_column_letter(3 + i)].width = 14

    metric = "Levered" if lev else "Unlevered"
    _title_bar(sv, f"{metric} IRR Sensitivity",
               "Purchase price (rows) vs. exit cap rate (columns)", width=2 + len(grid.columns))

    sv.cell(row=4, column=2, value="Price / Exit Cap").font = H2
    sv.cell(row=4, column=2).fill = FILL_NAVY
    for j, colname in enumerate(grid.columns):
        c = sv.cell(row=4, column=3 + j, value=colname)
        c.font = H2
        c.fill = FILL_NAVY
        c.alignment = Alignment(horizontal="center")

    for i, idxname in enumerate(grid.index):
        r = 5 + i
        c = sv.cell(row=r, column=2, value=idxname)
        c.font = LBL_B
        c.fill = FILL_GREY
        c.border = BOX
        for j in range(len(grid.columns)):
            v = sv.cell(row=r, column=3 + j, value=float(grid.values[i, j]))
            v.number_format = PCT2
            v.font = CALC
            v.border = BOX
            v.alignment = Alignment(horizontal="center")

    last_r = 4 + len(grid.index)
    last_c = get_column_letter(2 + len(grid.columns))
    sv.conditional_formatting.add(
        f"C5:{last_c}{last_r}",
        ColorScaleRule(start_type="min", start_color="F8696B",
                       mid_type="percentile", mid_value=50, mid_color="FFEB84",
                       end_type="max", end_color="63BE7B"),
    )
    sv.cell(row=last_r + 2, column=2,
            value="Base case is the center cell. Green indicates higher returns.").font = NOTE

    # =================================================================
    # DEAL SCREEN
    # =================================================================
    if ranked is not None and len(ranked):
        ds = wb.create_sheet("Deal Screen")
        ds.sheet_view.showGridLines = False
        ds.column_dimensions["A"].width = 3
        ds.column_dimensions["B"].width = 30
        for i in range(1, len(ranked.columns)):
            ds.column_dimensions[get_column_letter(2 + i)].width = 16

        _title_bar(ds, "Deal Screen", "All deals underwritten on identical assumptions, ranked by IRR",
                   width=1 + len(ranked.columns))

        fmts = {
            "Purchase Price": CUR0, "Going-In Cap": PCT2, "Exit Cap": PCT2,
            "Unlevered IRR": PCT2, "Levered IRR": PCT2,
            "Equity Multiple": MULT, "Yr1 DSCR": MULT,
        }
        for j, col in enumerate(ranked.columns):
            c = ds.cell(row=4, column=2 + j, value=col)
            c.font = H2
            c.fill = FILL_NAVY
            c.alignment = Alignment(horizontal="center")

        for i, (_, row) in enumerate(ranked.iterrows()):
            r = 5 + i
            for j, col in enumerate(ranked.columns):
                val = row[col]
                c = ds.cell(row=r, column=2 + j, value=val)
                c.font = LBL_B if j == 0 else CALC
                c.border = BOX
                if col in fmts:
                    c.number_format = fmts[col]
                    c.alignment = Alignment(horizontal="center")
            if i == 0:
                for j in range(len(ranked.columns)):
                    ds.cell(row=r, column=2 + j).fill = FILL_LIGHT

        ds.cell(row=5 + len(ranked) + 1, column=2,
                value="Top-ranked deal highlighted. Ranking is quantitative only.").font = NOTE

    # =================================================================
    # MEMO
    # =================================================================
    mw = wb.create_sheet("Investment Memo")
    mw.sheet_view.showGridLines = False
    mw.column_dimensions["A"].width = 3
    mw.column_dimensions["B"].width = 110

    _title_bar(mw, "Investment Memorandum", d["deal_name"], width=2)

    r = 4
    for block in memo_text.strip().split("\n\n"):
        block = block.strip()
        if not block:
            continue
        lines = block.split("\n")
        if lines[0].isupper() and len(lines[0]) < 60:
            c = mw.cell(row=r, column=2, value=lines[0])
            c.font = Font(name=FONT, size=11, bold=True, color=NAVY)
            r += 1
            body = " ".join(lines[1:]).strip()
        else:
            body = block
        if body:
            mw.merge_cells(start_row=r, start_column=2, end_row=r, end_column=2)
            c = mw.cell(row=r, column=2, value=body)
            c.font = LBL
            c.alignment = Alignment(wrap_text=True, vertical="top")
            mw.row_dimensions[r].height = max(30, 13 * (len(body) // 95 + 1))
            r += 2

    wb.save(path)
    return path


def _section_rows(ws, start, rows):
    for i, (label, val, fmt, note) in enumerate(rows):
        r = start + i
        ws.cell(row=r, column=2, value=label).font = LBL
        c = ws.cell(row=r, column=3, value=val)
        c.font = INPUT
        c.fill = FILL_INPUT
        c.number_format = fmt
        c.border = BOX
        ws.cell(row=r, column=4, value=note).font = NOTE


def export_package(results, grid, memo_text, ranked=None, folder="output"):
    Path(folder).mkdir(exist_ok=True)
    name = results["inputs"]["deal_name"].replace(" ", "_")

    xlsx_path = f"{folder}/{name}_Underwriting.xlsx"
    build_workbook(results, grid, ranked, memo_text, xlsx_path)

    docx_path = None
    try:
        from docx import Document
        from docx.shared import Pt
        doc = Document()
        st = doc.styles["Normal"]
        st.font.name = "Times New Roman"
        st.font.size = Pt(11)
        for block in memo_text.strip().split("\n\n"):
            block = block.strip()
            if not block:
                continue
            lines = block.split("\n")
            if lines[0].isupper() and len(lines[0]) < 60:
                h = doc.add_paragraph()
                h.add_run(lines[0]).bold = True
                if len(lines) > 1:
                    doc.add_paragraph(" ".join(lines[1:]))
            else:
                doc.add_paragraph(block)
        docx_path = f"{folder}/{name}_Investment_Memo.docx"
        doc.save(docx_path)
    except ImportError:
        print("python-docx not installed - skipping Word memo.")

    return xlsx_path, docx_path


xlsx, docx_out = export_package(
    results, grid,
    ai_memo if ai_memo else memo,
    ranked=ranked,
)
print("Wrote:", xlsx)
print("Wrote:", docx_out)
print()
print("The Excel workbook is a live model - edit the yellow input cells")
print("on the Assumptions tab and everything recalculates.")


---

### Notes on the AI component

The model never sees the raw deal file and never performs the math. Python computes every
figure; the language model only converts an audited block of numbers into prose, under an
explicit instruction not to invent facts. That division is deliberate — the arithmetic stays
reproducible and verifiable, and the model is confined to the task it is actually reliable at.

### Known limitations

- NOI grows at a single constant rate; no explicit lease-up, rollover, or capex schedule
- Terminal value is a forward-NOI cap, net of selling costs
- Debt is fixed-rate and fully amortizing; no interest-only period or refinancing
- No tax treatment, partnership waterfall, or promote structure

Appropriate for a screening-level base case. A deal that clears this screen still needs a
full property-level model.
